# Model Loading & Prediction — Churn Prediction

### 📌 Recap

From [03 Model Training.ipynb](03%20Model%20Training.ipynb): the trained ANN was saved (`model.h5`), alongside three pickled preprocessing artifacts from [02 Problem Statement.ipynb](02%20Problem%20Statement.ipynb) (`label_encoder_gender.pkl`, `onehot_encoder_geo.pkl`, `scaler.pkl`).

This notebook is the missing piece needed before deployment: given a **brand-new customer's raw data** (not from the training set), reload everything from disk and produce a churn prediction — exactly what the eventual Streamlit app will need to do on every user submission.

## 1. Load the trained model

> ⚠️ **Real bug found while testing this notebook — not in the video, but worth knowing:** calling `load_model("model.h5")` with no arguments fails here with a Keras internal deserialization error (`BinaryCrossentropy.__init__() got an unexpected keyword argument 'fn'`) — a known Keras 3 / HDF5 compatibility quirk when reloading the *compiled* optimizer+loss state, even within the exact same environment that saved it. The fix: **`compile=False`**. For pure prediction (no further training), the optimizer/loss state isn't needed anyway — only the architecture and trained weights are, and those load perfectly. If training needs to resume later, just call `model.compile(...)` again after loading (cheap, and avoids relying on fragile automatic config restoration).

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("model.h5", compile=False)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

> 📝 **Also worth knowing:** the `.keras` native format (saved alongside `.h5` in notebook 03, and recommended there as the "modern default") was tested here too — it fails to reload *at all* in this environment (`Layer 'dense' expected 2 variables, but received 0 variables during loading`), both with and without `compile=False`. So for this project, **`.h5` is actually the more reliable format in practice**, despite Keras's own deprecation warning recommending `.keras`. Worth keeping in mind if `03 Model Training.ipynb` gets revisited — the "modern format" isn't automatically the safer choice.

## 2. Load the encoders and scaler

In [2]:
import pickle

with open("label_encoder_gender.pkl", "rb") as file:
    label_encoder_gender = pickle.load(file)

with open("onehot_encoder_geo.pkl", "rb") as file:
    one_hot_encoder_geo = pickle.load(file)

with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

print("Loaded: label_encoder_gender, one_hot_encoder_geo, scaler")

Loaded: label_encoder_gender, one_hot_encoder_geo, scaler


C:\Users\ibhar\miniconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ibhar\miniconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ibhar\miniconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.7.1. This might lead to breaking

> ⚠️ **Another real gotcha hit while testing (environment-specific to this notebook series, but instructive):** loading these pickles here triggers `InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.7.1`. That's because notebook 02 (which created these pickles) ran on a different Python/scikit-learn installation than this notebook's dedicated TensorFlow kernel. It still works here, but this is *exactly* the class of problem the video's own project setup (**one dedicated conda environment for the whole pipeline**, from [01 Project Introduction.ipynb](01%20Project%20Introduction.ipynb)) is designed to prevent. **Lesson:** pickle files aren't just data — they're tied to the exact library version that created them; always load them back with a matching environment in a real project, not a mix of environments like this notebook series ended up needing.

## 3. New customer data to predict on

A single new customer's raw information — exactly the shape of data the Streamlit app's form fields will eventually collect.

In [3]:
import pandas as pd

input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000,
}

input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


## 4. Apply the same preprocessing as training

### 4a. One-hot encode `Geography`

> 💡 **The same 2D-array gotcha from [notebook 02](02%20Problem%20Statement.ipynb):** `one_hot_encoder_geo.transform(input_df["Geography"])` fails with *"Expected 2D array, got 1D array instead"*, because `input_df["Geography"]` is a 1D Series. The fix is the same one used during training: select with **double brackets**, `input_df[["Geography"]]`, which keeps it a 2D DataFrame (1 row × 1 column) — matching what the encoder was fit on.

In [4]:
geo_encoded = one_hot_encoder_geo.transform(input_df[["Geography"]])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encoder_geo.get_feature_names_out(["Geography"]),
)
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


### 4b. Label encode `Gender`, then combine everything

In [5]:
input_df["Gender"] = label_encoder_gender.transform(input_df["Gender"])

input_df = pd.concat(
    [input_df.drop("Geography", axis=1).reset_index(drop=True), geo_encoded_df],
    axis=1,
)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


## 5. ⚠️ Correction: column order matters for scaling — and the failure mode is silent

The video doesn't flag this, but it's a real and important risk verified directly against the actual pickled `scaler`. `StandardScaler` learns per-column mean/std during `fit`, in a fixed column **order** — `scaler.transform()` must receive columns in that exact same order, or every value gets scaled with the *wrong* column's statistics.

Testing this directly against the real `scaler.pkl`:

| Input type | Column order | Result |
|---|---|---|
| pandas DataFrame | correct | works |
| pandas DataFrame | **wrong** | **raises `ValueError`** — scikit-learn checks column *names* against `scaler.feature_names_in_` and refuses to proceed |
| plain NumPy array | **wrong** | **silently returns wrong numbers, no error or warning at all** |

So keeping `input_df` as a named DataFrame (as done here) gets a safety net "for free" — but it's still worth being explicit and enforcing the order directly, rather than relying on the columns having *happened* to end up in the right sequence after the concat above. `scaler.feature_names_in_` (saved automatically during `fit`) is exactly the reference order to use.

In [6]:
print("Expected column order (from training):", list(scaler.feature_names_in_))
print("Current column order            :", list(input_df.columns))

# enforce the exact training order explicitly, rather than trusting it fell out that way
input_df = input_df[scaler.feature_names_in_]
print("\nReordered to match - safe to scale now.")

Expected column order (from training): ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']
Current column order            : ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']

Reordered to match - safe to scale now.


## 6. Scale the input

In [7]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

## 7. Predict

`model.predict` returns a probability (from the output layer's sigmoid activation) that this customer churns.

In [8]:
prediction = model.predict(input_scaled, verbose=0)
prediction_probability = prediction[0][0]

print(f"Churn probability: {prediction_probability:.4f}")

if prediction_probability > 0.5:
    print("Prediction: the customer is LIKELY to churn.")
else:
    print("Prediction: the customer is NOT likely to churn.")

Churn probability: 0.0338
Prediction: the customer is NOT likely to churn.


## 8. Summary

- Reload order: **model** (`load_model("model.h5", compile=False)`, needed to sidestep a real Keras deserialization bug) → **encoders + scaler** (`pickle.load`).
- New raw customer data → DataFrame → one-hot encode `Geography` (`[["Geography"]]`, keeping it 2D) → label encode `Gender` → concatenate → **enforce column order to match `scaler.feature_names_in_`** → scale → `model.predict` → threshold at 0.5.
- This exact sequence — load artifacts, preprocess one new row the same way training data was preprocessed, scale, predict — is precisely what the Streamlit app will do per user submission.

**Real issues found and fixed while building this notebook (none flagged in the video):**
1. `load_model("model.h5")` fails without `compile=False`, even in the same environment/session that saved it — a Keras 3 HDF5 loss-deserialization bug.
2. The `.keras` format (recommended in notebook 03 as the "modern default") actually fails to reload *at all* here — `.h5` turned out to be the more reliable format for this project in practice.
3. Loading pickles created in a different Python/scikit-learn install triggers an `InconsistentVersionWarning` — a real illustration of why the video insists on one dedicated project environment.
4. **Column-order mismatch during scaling fails loudly for a DataFrame, but silently for a plain NumPy array** — verified directly against the real `scaler.pkl`. Worth remembering when building the Streamlit form, where it'll be tempting to assemble inputs as a plain list/array.

## 9. Likely exam / interview questions

1. Why is `compile=False` sometimes necessary (and harmless) when loading a saved Keras model purely for inference?
2. Why must new input data be one-hot/label encoded with the *same fitted* encoder objects used during training, rather than fitting new ones?
3. What happens if a column is missing or misnamed when calling `scaler.transform()` on a DataFrame? What about on a plain NumPy array?
4. Why does `input_df[["Geography"]]` work for `OneHotEncoder.transform()` but `input_df["Geography"]` doesn't?
5. What does the model's raw prediction actually represent, and why is 0.5 used as the decision threshold?
6. Why is it good practice to keep one dedicated environment for an entire ML project pipeline?

## 10. What's next

- Build the **Streamlit** web app: a form collecting the same 10 raw fields used here, wired through this exact preprocessing → scaling → prediction pipeline.
- Deploy the app to **Streamlit Cloud**.